# clue_rl

> Phase 12, Track B: RL on the clue board (GRPO with one shared adapter, or one per agent slot). Per step, new boards are each played `group` times with the current model; every slot's return (the reward scheme's, minus a training-only penalty for invalid replies) is compared with its group's; one policy-gradient update with a KL penalty to the warm-start model; checkpoints with exact resume (`rl_torch`). Training needs the `[train]` extra; the offline tests do not.

In [ ]:
#| default_exp clue_rl

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib
import json
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np

from explRL.clue_grid import BOARD, CLUE_SETTINGS
from explRL.clue_roll import coopRows, rollout
from explRL.clue_sft import libVersions, loadModel, loraConfig, saveMemory
from explRL.llm_runs import LLMConfig
from explRL.local_policy import LocalPolicy, pickDevice
from explRL.rl_torch import loadCheckpoint, policyUpdate, saveCheckpoint


@dataclass
class RLConfig:
    model: str                  # the warm-start model (merged weights)
    out: str
    scheme: str = 'credit'
    N: int = 4
    channel: str = 'blackboard'
    turns: int = 10
    seedsPerStep: int = 8
    group: int = 4
    steps: int = 40
    lr: float = 1e-5
    kl: float = 0.04
    loraR: int = 16
    loraAlpha: int = 32
    maxNew: int = 160
    temperature: float = 1.0
    formatPenalty: float = 0.1
    trainSeed: int = 0
    boardSeed0: int = 10_000
    ckptEvery: int = 5
    microBatch: int = 8
    maxLen: int = 1536
    device: str = 'auto'
    perRole: bool = False       # one adapter per agent slot (the optional ablation)
    gradCkpt: bool = True       # gradient checkpointing (less memory, about a third more compute)
    turnFraction: float = 0.25  # train on this share (seeded per step) of the turns with a nonzero advantage
    haltInvalid: float = 0.10   # stopping rule: invalid-reply rate above this for haltRun steps in a row ...
    haltRun: int = 3
    haltKL: float = 0.5         # ... or the mean KL above this


def configHash(cfg):
    "Identity of a training run: everything but where it runs and how long (a run can be resumed and extended)."
    d = {k: v for k, v in asdict(cfg).items() if k not in ('out', 'device', 'ckptEvery', 'microBatch', 'steps', 'gradCkpt')}
    return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:12]


def adapterNames(cfg):
    return [f'slot{i}' for i in range(cfg.N)] if cfg.perRole else ['default']


def stepSeeds(cfg, s):
    "Board seeds of step s: seedsPerStep new boards, each played `group` times."
    return [cfg.boardSeed0 + s * cfg.seedsPerStep + j for j in range(cfg.seedsPerStep) for _ in range(cfg.group)]


def trainingReturns(ro, penalty):
    "Rollout returns minus `penalty` per invalid reply of the slot (training only; reported welfare is unchanged)."
    R = np.array(ro.returns, float)
    for b, lg in enumerate(ro.logs):
        for i, recs in enumerate(lg):
            R[b, i] -= penalty * sum(not x['valid'] for x in recs)
    return R


def groupAdvantages(returns):
    "returns (seeds x group x N) -> per slot, (R - group mean) / (group std + 1e-6); 0 where the group's returns are equal."
    R = np.asarray(returns, float)
    mu, sd = R.mean(axis=1, keepdims=True), R.std(axis=1, keepdims=True)
    return np.where(sd > 1e-9, (R - mu) / (sd + 1e-6), 0.0)


def stepSamples(ro, adv, fraction=1.0, rng=None):
    """One sample per text turn of every slot with a nonzero advantage (slot, prompt, reply text, cut or not, advantage);
    with fraction < 1, a random share of them (an unbiased, cheaper estimate of the same gradient)."""
    out = [dict(slot=i, system=p[0], blocks=p[1], reply=x['response'], eos=x['stop'] != 'max_tokens', adv=float(adv[b, i]))
           for b in range(len(ro.boards)) for i, (recs, ps) in enumerate(zip(ro.logs[b], ro.prompts[b])) if adv[b, i] != 0
           for x, p in zip(recs, ps)]
    if fraction >= 1 or not out:
        return out
    keep = sorted((rng or np.random.default_rng(0)).choice(len(out), max(1, round(fraction * len(out))), replace=False))
    return [out[k] for k in keep]


def haltReason(history, cfg):
    "The stopping rule: invalid replies above haltInvalid for haltRun steps in a row, or the mean KL above haltKL."
    last = history[-cfg.haltRun:]
    if len(last) == cfg.haltRun and all(h['invalid'] > cfg.haltInvalid for h in last):
        return f"invalid replies above {cfg.haltInvalid:g} for {cfg.haltRun} steps (steps {last[0]['step']}-{last[-1]['step']})"
    if history and history[-1]['kl'] > cfg.haltKL:
        return f"mean KL {history[-1]['kl']:.3f} above {cfg.haltKL:g} at step {history[-1]['step']}"
    return None


def microBatches(samples, cfg):
    "Chunks of at most microBatch samples; with per-role adapters every chunk holds one slot's samples."
    key = (lambda x: x['slot']) if cfg.perRole else (lambda x: 0)
    groups = [[x for x in samples if key(x) == k] for k in sorted({key(x) for x in samples})]
    return [g[a:a + cfg.microBatch] for g in groups for a in range(0, len(g), cfg.microBatch)]


def _mean(x):
    x = [v for v in x if not np.isnan(v)]
    return float(np.mean(x)) if x else float('nan')


def trainRL(cfg, settings=CLUE_SETTINGS):
    "GRPO on the clue board from the warm-start model (one update per step, KL to the adapter-free model); -> final adapter."
    import torch
    from peft import get_peft_model
    out, dev = Path(cfg.out), pickDevice(cfg.device)
    out.mkdir(parents=True, exist_ok=True)
    (out / 'config.json').write_text(json.dumps(dict(asdict(cfg), hash=configHash(cfg), versions=libVersions()), indent=1) + '\n')
    torch.manual_seed(cfg.trainSeed)
    tok, base = loadModel(cfg.model, dev)
    names = adapterNames(cfg)
    model = get_peft_model(base, loraConfig(cfg.loraR, cfg.loraAlpha), adapter_name=names[0])
    for n in names[1:]:
        model.add_adapter(n, loraConfig(cfg.loraR, cfg.loraAlpha))
    saveMemory(model, cfg.gradCkpt)
    params = [p for n, p in model.named_parameters() if 'lora_' in n]
    for p in params:
        p.requires_grad_(True)
    opt = torch.optim.AdamW(params, lr=cfg.lr)
    start = loadCheckpoint(model, opt, out, configHash(cfg), names)
    if (out / 'halted.json').exists():                       # stopped by the stopping rule: never silently resumed
        print('halted:', json.loads((out / 'halted.json').read_text())['reason'])
        return str(out / 'final')
    history = [h for h in map(json.loads, open(out / 'metrics.jsonl')) if h['step'] <= start] if (out / 'metrics.jsonl').exists() else []
    vllm = dev == 'cuda' and LocalPolicy(cfg.model, device=dev, engine='auto', tokenizer=tok).engine == 'vllm'
    first = LocalPolicy(cfg.model, device=dev, maxNew=cfg.maxNew, temperature=cfg.temperature, seed=cfg.trainSeed,
                        engine='vllm' if vllm else 'hf', hf=None if vllm else model, tokenizer=tok, adapterName=names[0])
    views = [first] + [LocalPolicy(cfg.model, device=dev, maxNew=cfg.maxNew, temperature=cfg.temperature,
                                   seed=cfg.trainSeed, engine=first.engine, hf=first.hf, tokenizer=tok, adapterName=n,
                                   share=first) for n in names[1:]]
    pol = {i: views[i] for i in range(cfg.N)} if cfg.perRole else first
    lcfg = LLMConfig(task='clue', channel=cfg.channel, N=cfg.N, turns=cfg.turns, model=cfg.model,
                     world={**{k: settings[k] for k in BOARD}, 'scheme': cfg.scheme})
    with open(out / 'metrics.jsonl', 'a') as log:
        for s in range(start, cfg.steps):
            t0 = time.time()
            model.eval()
            for v, n in zip(views, names):
                v.calls = s * 100_000                       # sampling seeds depend on the step only (exact resume)
                if vllm:
                    model.save_pretrained(out / 'live')
                    v.setAdapter(str(out / 'live' / ('' if n == 'default' else n)))
            ro = rollout(lcfg, stepSeeds(cfg, s), pol, keepPrompts=True)
            R = trainingReturns(ro, cfg.formatPenalty)
            adv = groupAdvantages(R.reshape(cfg.seedsPerStep, cfg.group, cfg.N)).reshape(R.shape)
            signal = stepSamples(ro, adv)
            samples = stepSamples(ro, adv, cfg.turnFraction, np.random.default_rng([cfg.trainSeed, s]))
            t1 = time.time()
            model.train()
            stats = policyUpdate(model, tok, opt, params, samples, microBatches(samples, cfg), names, cfg.perRole, cfg.kl,
                                 cfg.maxLen, dev)
            rows = coopRows(lcfg, ro)
            m = dict(step=s + 1, rolloutSeconds=round(t1 - t0, 2), updateSeconds=round(time.time() - t1, 2),
                     meanReturn=float(R.mean()), welfare=float(np.mean([r['welfare'] for r in rows])),
                     sharePerHeld=float(np.mean([r['sharePerHeld'] for r in rows])),
                     invalid=sum(r['invalid'] for r in rows) / max(sum(r['calls'] for r in rows), 1),
                     soloShare=_mean([r['soloShare'] for r in rows]), signalTurns=len(signal), samples=len(samples), **stats)
            log.write(json.dumps(m) + '\n')
            log.flush()
            history.append(m)
            reason = haltReason(history, cfg)
            if (s + 1) % cfg.ckptEvery == 0 or s + 1 == cfg.steps or reason:
                saveCheckpoint(model, opt, s + 1, out, configHash(cfg))
                (out / 'turns').mkdir(exist_ok=True)       # this step's turns, to read what the agents do
                (out / 'turns' / f'step-{s + 1:04d}.jsonl').write_text(''.join(
                    json.dumps(dict(board=b, seed=ro.seeds[b], **x), sort_keys=True) + '\n'
                    for b in range(len(ro.boards)) for lg in ro.logs[b] for x in lg))
            if reason:
                (out / 'halted.json').write_text(json.dumps(dict(step=s + 1, reason=reason)) + '\n')
                print('halted:', reason)
                break
    model.save_pretrained(out / 'final')
    return str(out / 'final')

## Tests

In [ ]:
import numpy as np
from types import SimpleNamespace
from explRL.clue_policy import claudePolicy
from explRL.clue_roll import rollout
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_runs import LLMConfig

# AC-5: group advantages on hand-made returns (seeds x group x N); equal returns give 0
R = np.array([[[1.0, 0.0], [3.0, 0.0]], [[2.0, 5.0], [2.0, 1.0]]])
A = groupAdvantages(R)
assert np.allclose(A[0, :, 0], [-1, 1], atol=1e-5) and np.all(A[0, :, 1] == 0) and np.all(A[1, :, 0] == 0)
assert np.allclose(A[1, :, 1], [1, -1], atol=1e-5)
# the format penalty lowers training returns only, per invalid reply of the slot
ro = SimpleNamespace(returns=np.array([[1.0, 2.0]]), logs=[[[dict(valid=True), dict(valid=False)], [dict(valid=False)] * 3]])
assert np.allclose(trainingReturns(ro, 0.1), [[0.9, 1.7]]) and np.allclose(ro.returns, [[1.0, 2.0]])
# samples: one per text turn of slots with nonzero advantage, with its prompt; zero-advantage slots skipped
cfg = LLMConfig(task='clue', channel='blackboard', N=4, turns=3, world=dict(m=2, r=1, R=30))
ro = rollout(cfg, [900, 900], claudePolicy(LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))), keepPrompts=True)
adv = np.zeros((2, 4)); adv[0, 1], adv[1, 3] = 0.5, -1.0
smp = stepSamples(ro, adv)
assert len(smp) == 6 and {x['slot'] for x in smp} == {1, 3} and all(x['blocks'][-1]['text'].startswith('Now:\n') for x in smp)
assert [x['adv'] for x in smp] == [0.5] * 3 + [-1.0] * 3 and all(x['eos'] for x in smp)
# a share of the turns: a seeded, sorted subset of the full set, of the requested size
full = stepSamples(ro, adv)
part = stepSamples(ro, adv, 0.5, np.random.default_rng([0, 7]))
assert len(part) == 3 and all(x in full for x in part) and part == stepSamples(ro, adv, 0.5, np.random.default_rng([0, 7]))
assert stepSamples(ro, adv * 0, 0.25) == [] and len(stepSamples(ro, adv, 0.01)) == 1
# the stopping rule: invalid replies too high for haltRun steps in a row, or the KL too high
c = RLConfig('m', 'o')
h = lambda *xs: [dict(step=k + 1, invalid=v, kl=kl) for k, (v, kl) in enumerate(xs)]
assert haltReason(h((0.2, 0.0), (0.2, 0.0)), c) is None and haltReason(h((0.2, 0.0), (0.05, 0.0), (0.2, 0.0), (0.2, 0.0)), c) is None
assert 'steps 2-4' in haltReason(h((0.0, 0.0), (0.2, 0.0), (0.3, 0.0), (0.11, 0.0)), c)
assert 'KL 0.600' in haltReason(h((0.0, 0.1), (0.0, 0.6)), c) and haltReason([], c) is None
# seeds: new boards every step, each played `group` times; configuration identity
c = RLConfig('m', 'o', seedsPerStep=2, group=3)
assert stepSeeds(c, 0) == [10000] * 3 + [10001] * 3 and stepSeeds(c, 1)[0] == 10002
assert configHash(c) == configHash(RLConfig('m', 'elsewhere', seedsPerStep=2, group=3, device='cpu', steps=80, gradCkpt=False))
assert configHash(c) != configHash(RLConfig('m', 'o', seedsPerStep=2, group=3, lr=2e-5))
# per-role adapters: one per slot; micro-batches never mix slots
assert adapterNames(RLConfig('m', 'o')) == ['default'] and adapterNames(RLConfig('m', 'o', perRole=True)) == ['slot0', 'slot1', 'slot2', 'slot3']
mb = microBatches([dict(slot=s) for s in (0, 1, 0, 1, 1)], RLConfig('m', 'o', perRole=True, microBatch=2))
assert [[x['slot'] for x in b] for b in mb] == [[0, 0], [1, 1], [1]]
assert [len(b) for b in microBatches([dict(slot=s) for s in (0, 1, 0, 1, 1)], RLConfig('m', 'o', microBatch=2))] == [2, 2, 1]

In [ ]:
#| train
# With the [train] extra, no download: two RL steps on the tiny random model (whose replies are all invalid, so random
# returns stand in to give nonzero advantages), then resume at step 3, a refused configuration, and per-role adapters
import tempfile
from pathlib import Path
from explRL.tiny_model import tinyModel
_rng, _tr = np.random.default_rng(0), trainingReturns   # trainRL here looks it up in this namespace
trainingReturns = lambda ro, p: _tr(ro, p) + _rng.normal(size=np.shape(ro.returns))
try:
    with tempfile.TemporaryDirectory() as d:
        tiny = tinyModel(d + '/tiny')
        for perRole in (False, True):
            cfg = RLConfig(tiny, f'{d}/rl{perRole}', steps=2, seedsPerStep=2, group=3, turns=3, ckptEvery=1, maxNew=24, perRole=perRole, lr=1e-3)
            trainRL(cfg)
            trainRL(RLConfig(**{**asdict(cfg), 'steps': 3}))
            m = [json.loads(x) for x in open(f'{d}/rl{perRole}/metrics.jsonl')]
            assert [x['step'] for x in m] == [1, 2, 3] and all(x['samples'] > 0 and np.isfinite(x['loss']) for x in m), m
            assert m[0]['kl'] == 0 and m[-1]['kl'] > 0                    # the fresh adapter starts at the warm-start model
            assert sorted(p.name for p in Path(f'{d}/rl{perRole}/turns').iterdir()) == ['step-0001.jsonl', 'step-0002.jsonl', 'step-0003.jsonl']
            assert json.load(open(f'{d}/rl{perRole}/config.json'))['versions']['torch']
            try:
                trainRL(RLConfig(**{**asdict(cfg), 'lr': 2e-3, 'steps': 4}))
                raise AssertionError('a checkpoint of another configuration must be refused')
            except ValueError:
                pass
            # the tiny model's replies are all invalid: the stopping rule halts at step 3, and a halted run is not resumed
            assert 'invalid replies' in json.load(open(f'{d}/rl{perRole}/halted.json'))['reason']
            trainRL(RLConfig(**{**asdict(cfg), 'steps': 5}))
            assert len(open(f'{d}/rl{perRole}/metrics.jsonl').readlines()) == 3
finally:
    trainingReturns = _tr

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()